### Import Dependencies

In [1]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

/Users/wasanthagamage/Documents/ai-bootcamp/ai-engineering-bootcamp-cohort-5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Download an example reference data point from LangSmith

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [3]:
ls_client = Client()

In [4]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [5]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('2f29c115-0e54-4db7-aa57-305b25316bb1'), created_at=datetime.datetime(2026, 7, 5, 17, 35, 23, 495576, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 5, 17, 35, 23, 495576, tzinfo=TzInfo(0)), example_count=32, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'sdk_version': '0.8.16', 'runtime_version': '3.12.13', 'langchain_version': '1.3.2', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.7'}})

In [6]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': 'You have two main options. One is a single 1m or 3.3ft MFi-certified Lightning cable. The other is a 3-pack of 6ft MFi-certified Lightning cables, which is better if you want longer cables or extras for different rooms.',
 'reference_context_ids': ['B0BN1CMWCP', 'B0BBVJJRHD'],
 'reference_descriptions': ["iPhone Charger Cord [Apple MFi Certified] Lightning Cable 1m/3.3FT Fast Charging High Speed Data Sync USB Cable Compatible with iPhone 14/13/12/11 Pro Max/XS MAX/XR/XS/X/8/7/Plus/6S iPad [Fast Charging & Sync] - iPhone Charger Cord Max 2.4 output current, high quality copper wire, improve charging and data transfer speed of iPhone charging cable. iPhone Lightning Cable ensures faster charging time and keeps your device safer. [Perfect comptibility] - iPhone lightning cables compatible with iPhone 14/14 Plus/14 Pro/14 Pro Max/iPhone 13/13 mini/13 Pro/13 Pro Max/12/12 mini/12 Pro/12 Pro Max/11/11 Pro/11 Pro Max/SE 2/XS/XS Max/XR/X/8Plus/8/7Plus/7/6s Plus/6s/6 Plus/6/5s

In [7]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': 'What iPhone charging cable options do you have if I want either a shorter single cable or a longer multi-pack?'}

In [8]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

### RAG Pipeline

In [10]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [11]:
rag_pipeline("Can I get a charger?")

{'answer': 'Yes. We have a few charger options available:\n\n1) iPhone Charger 6ft 3Pack (Apple MFi certified Lightning cables) – supports fast charging up to 3A and data sync (480Mbps), reinforced durable design, compatible with many iPhone and iPad models.\n\n2) USB C to USB C Cable, INIU 6.6ft (100W PD 5A) – fast, high-power USB-C cable compatible with USB-C phones/tablets and many laptops (including MacBook Pro), with Emark 2.0 chip for safety.\n\n3) Notebook Charger/Replacement Charger (White-60W) – compatible with notebook charging; note says this is the second generation and you should check your exact Mac notebook model before buying.\n\nWhich device are you charging (iPhone with Lightning, USB-C device, or a Mac notebook model)?',
 'question': 'Can I get a charger?',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BGH3H1WM',
  'B0BN1CMWCP',
  'B0C9QZS95R',
  'B0BXC72RLD'],
 'retrieved_context': ['iPhone Charger 6ft 3Pack, Apple MFi Certified Lightning Cable Fast Charging Long iP

### RAGAS Metrics

In [12]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/h2/_7l4lv3546966sccvj1yfdq00000gn/T/ipykernel_74717/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/h2/_7l4lv3546966sccvj1yfdq00000gn/T/ipykernel_74717/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/h2/_7l4lv3546966sccvj1yfdq00000gn/T/ipykernel_74717/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [13]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/h2/_7l4lv3546966sccvj1yfdq00000gn/T/ipykernel_74717/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/h2/_7l4lv3546966sccvj1yfdq00000gn/T/ipykernel_74717/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [14]:
reference_input

{'question': 'What iPhone charging cable options do you have if I want either a shorter single cable or a longer multi-pack?'}

In [15]:
reference_output

{'ground_truth': 'You have two main options. One is a single 1m or 3.3ft MFi-certified Lightning cable. The other is a 3-pack of 6ft MFi-certified Lightning cables, which is better if you want longer cables or extras for different rooms.',
 'reference_context_ids': ['B0BN1CMWCP', 'B0BBVJJRHD'],
 'reference_descriptions': ["iPhone Charger Cord [Apple MFi Certified] Lightning Cable 1m/3.3FT Fast Charging High Speed Data Sync USB Cable Compatible with iPhone 14/13/12/11 Pro Max/XS MAX/XR/XS/X/8/7/Plus/6S iPad [Fast Charging & Sync] - iPhone Charger Cord Max 2.4 output current, high quality copper wire, improve charging and data transfer speed of iPhone charging cable. iPhone Lightning Cable ensures faster charging time and keeps your device safer. [Perfect comptibility] - iPhone lightning cables compatible with iPhone 14/14 Plus/14 Pro/14 Pro Max/iPhone 13/13 mini/13 Pro/13 Pro Max/12/12 mini/12 Pro/12 Pro Max/11/11 Pro/11 Pro Max/SE 2/XS/XS Max/XR/X/8Plus/8/7Plus/7/6s Plus/6s/6 Plus/6/5s

In [16]:
result = rag_pipeline(reference_input["question"])

In [17]:
result

{'answer': 'You have these iPhone Lightning charging cable options:\n\n1) Shorter single cable\n- 1x cable, 3.3 ft / 1m length\n- “Apple MFi Certified” Lightning cable\n- Fast charging and high-speed data sync (max 2.4 output current)\n- Rating: 4.4\n\n2) Longer multi-pack\n- 3-pack of cables, each 6 ft long\n- “Apple MFi Certified” Lightning cables\n- Fast charging (up to 3A) and stable data transfer (480Mbps)\n- Rating: 4.3',
 'question': 'What iPhone charging cable options do you have if I want either a shorter single cable or a longer multi-pack?',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BN1CMWCP',
  'B0CC4HBS85',
  'B0C9QZS95R',
  'B0CH8DRD6K'],
 'retrieved_context': ['iPhone Charger 6ft 3Pack, Apple MFi Certified Lightning Cable Fast Charging Long iPhone Charger Cord High Speed Data Sync Cable Compatible iPhone 14 13 12 11 Pro Max XS XR X 8 7 6S 6 Plus SE 5S, iPad 【iPhone Cable Fast Charging】:The 8-pin connector with lightning end ensures safe charging of your iPhone device

In [18]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [19]:
await ragas_context_precision_id_based(result, reference_output)

0.4

In [20]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [21]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [22]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [23]:
await ragas_faithfulness(result, reference_output)

0.6666666666666666

In [24]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [25]:
await ragas_relevancy(result, reference_output)

np.float64(0.7932148247121438)